# Lesson 2 exercise — One training step by hand

**Goal:** write the PyTorch training loop yourself. Train `SmolLM2-135M-Instruct` on **one** example until it answers the Lesson 1 question with exactly `{"name": "Sara", "city": "Cairo"}`.

**How to use this notebook**
1. In Colab: *Runtime → Change runtime type → T4 GPU*.
2. Run cells top to bottom. Wherever you see `...` after a `# TODO`, replace it with your code.
3. Run the **check** cell under each TODO. ✅ means you've got it; ❌ gives you a hint.

Stuck? Look up the error in the course's troubleshooting sheet (`reference/troubleshooting.html`). Still stuck? Ask on the Hugging Face Forums with your code and the full traceback.

## 0 · Setup
Installs the course's library versions (about a minute) and defines the `check` helper.

In [ ]:
%pip install -q -U "transformers>=5.19,<6" "trl>=1.15,<2" "peft>=0.21,<1" "datasets>=5,<6" "accelerate>=1.15,<2"
import torch, transformers
device = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| torch", torch.__version__, "| device:", device)

def check(passed, ok_msg, hint):
    print(("✅ " + ok_msg) if passed else ("❌ " + hint))

## 1 · The model that failed in Lesson 1
Same checkpoint as Lesson 1, loaded in full precision (fp32). The T4 has no native bf16, and a 135M model is small enough to train in fp32.

`ask()` is Lesson 1 wrapped in a function: template → `generate()` → decode only the new tokens.

In [ ]:
import json, time
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32).to(device)

def ask(messages):
    inputs = tok.apply_chat_template(messages, add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True).to(device)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=40, do_sample=False)
    return tok.decode(out_ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

messages = [{"role": "user", "content": "Extract the person's name and city as JSON: 'Hi, I'm Sara from Cairo.'"}]
target = json.dumps({"name": "Sara", "city": "Cairo"})      # the exact reply we want

print("target:", target)
print("before:", repr(ask(messages)))

## 2 · The training example
Supervised fine-tuning (SFT) data is the conversation you *wish* had happened: the user's message plus the ideal assistant reply. It goes through the same chat template. There's no `add_generation_prompt=True` here, because the assistant turn is already in the messages.

In [ ]:
train_messages = messages + [{"role": "assistant", "content": target}]
print(tok.apply_chat_template(train_messages, tokenize=False))

batch = tok.apply_chat_template(train_messages, return_tensors="pt", return_dict=True).to(device)
print("input_ids shape:", tuple(batch["input_ids"].shape))      # (batch, seq)

The text ends with `<|im_end|>` after the JSON, so training on it also teaches the model to **stop**.

## 3 · Let the model compute its loss
Hugging Face causal-LM models compute the loss themselves when you pass `labels`. You don't shift anything: inside the model, the prediction at position *t* is scored against the token at position *t + 1*.

**TODO 1:** Set `labels` to the tokens the model should learn to predict.

In [ ]:
# TODO 1
labels = ...

out = model(**batch, labels=labels)
print("loss:", out.loss)

In [ ]:
# check 1
ok = torch.is_tensor(labels) and torch.equal(labels, batch["input_ids"]) and out.loss is not None
check(ok, f"Loss {out.loss.item():.4f}: one number for the whole sequence, ready for backward()." if ok else "",
      "No shifting needed. Which tensor in `batch` already holds every token of the training text?")

**No magic.** That loss is plain cross-entropy, the same loss as a Keras classifier, applied at every position. Compute it by hand: the scores made at positions 0 … n−2 against the tokens that actually came next (positions 1 … n−1).

In [ ]:
import torch.nn.functional as F

scores = out.logits[0, :-1]                # (seq-1, vocab): a prediction at each position...
next_tokens = batch["input_ids"][0, 1:]    # (seq-1,): ...and the token that really came next
by_hand = F.cross_entropy(scores, next_tokens)
print(f"model: {out.loss.item():.4f} | by hand: {by_hand.item():.4f}")

This loss counts **every** token, including the system prompt and the question. Lesson 3 fixes that.

## 4 · The training loop
This is what `model.fit()` did for you. You create the optimizer and hand it the tensors it may update. Then each step is: forward pass → gradients → update → clear the gradients.

**TODO 2:** Write the three lines that turn the loss into a weight update, in this order:
1. compute the gradients of the loss,
2. let the optimizer update the weights,
3. clear the gradients (PyTorch *adds* new gradients to old ones).

It takes a few seconds on a GPU. Watch the loss fall.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

model.train()                       # training mode (from_pretrained gave you eval mode)
losses, t0 = [], time.time()
for step in range(30):
    out = model(**batch, labels=labels)
    # TODO 2: three lines
    ...
    losses.append(out.loss.item())
    if step % 5 == 0 or step == 29:
        print(f"step {step:2d}   loss {out.loss.item():.4f}")
print(f"{time.time() - t0:.1f} s")

In [ ]:
# check 2
ok = losses[-1] < 0.05 and all(p.grad is None for p in model.parameters())
check(ok, f"Loss fell from {losses[0]:.2f} to {losses[-1]:.4f}, and the gradients are cleared.",
      "The loss should end below 0.05, with the gradients cleared as the last line of each step. "
      "After editing the loop, rerun section 1 to get a fresh model, then rerun the loop.")

## 5 · Did it learn?
Generating is inference, so switch back to eval mode first. `model.eval()` turns off training-only behaviour such as dropout. (It doesn't stop gradient tracking: `ask()` uses `torch.no_grad()` for that.)

**TODO 3:** Put the model in eval mode, then get its reply to `messages` with `ask()`.

In [ ]:
# TODO 3
...
reply = ...

print(repr(reply))

In [ ]:
# check 3
check(not model.training and reply == target,
      "Exactly the target JSON. Your loop fixed what Lesson 1 couldn't.",
      "Is the model in eval mode (look at model.training)? If the reply is close but not exact, run the loop cell again.")

## Stretch (optional)
1. Ask about someone else: `ask([{"role": "user", "content": "Extract the person's name and city as JSON: 'Hello, Omar here, I live in Paris.'"}])`. Did one example teach a *format*, or only one answer?
2. Rerun section 1 for a fresh model. Delete the line that clears the gradients, then rerun the loop. What happens to the loss?
3. Fresh model again, with `lr=1e-3`. What does the loss do?